# ⚖️ 20. Validación estadística del modelo original

Este capítulo responde la condición **R3**: ¿el ACS supera a los benchmarks de forma estadísticamente significativa? La guía define la superioridad con **tres criterios simultáneos**: (1) mejor métrica media del bucle externo que el mejor benchmark y que la mejor combinación de cada uno de los 7 modelos; (2) al menos tres pruebas de hipótesis distintas que rechacen tras ajustar por comparaciones múltiples (α = 0,05; 0,10 para el MCS); (3) un tamaño de efecto no despreciable.

**Métrica de la decisión.** La métrica principal del proyecto es el **RMSE en log**, fijada en la entrega 2 (cap. 9.6) **antes** de diseñar el ACS. Se reporta también el **MAE en log** (secundaria, presente desde la entrega 2), pero el veredicto se toma con la métrica fijada de antemano: elegir la métrica después de ver los resultados invalidaría la prueba.

In [1]:
import pickle
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller

sys.path.insert(0, str(Path.cwd().parent / "src"))
from comun import estilo, DATA, SEED, a_log, AZUL, NARANJA, VERDE, GRIS
from estadistica import diebold_mariano, ic_estacionario, cohen_d, cliff_delta, magnitud_cliff, friedman_nemenyi
from validacion_original import cargar_todo, pruebas

warnings.filterwarnings("ignore")
estilo()
train, test, sub, y, fold, preds, variantes, acs = cargar_todo()
RIVAL = "Random Forest · Grid"
print(f"{len(preds)} modelos en el universo de comparación; {len(y):,} negocio-días de validación externa en {sub['fecha'].nunique()} días")

/opt/miniconda3/envs/ml_venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


35 modelos en el universo de comparación; 12,436 negocio-días de validación externa en 28 días


## 20.1 Protocolo

- **Pareado estricto:** todos los modelos se evalúan en **las mismas filas** de validación externa (los 4 folds de 7 días de la entrega 2), con la misma semilla y el mismo preprocesamiento (pipeline del cap. 7). El ACS pasó por la misma validación anidada (cap. 19.7).
- **Universo:** 28 combinaciones de la entrega 2, 4 de HistGradientBoosting (estado del arte), el ACS y 2 líneas base: **35 modelos**.
- **Dos tipos de dependencia.** Los errores de un mismo día están correlacionados entre negocios y los de un mismo negocio entre días. Por eso se usan dos familias de pruebas: **series diarias** de pérdida (28 días; MCS, Diebold-Mariano, Giacomini-White, Clark-West, SPA, bootstrap estacionario) y **corte transversal por negocio** (447 negocios, unidades independientes; Wilcoxon, permutación pareada, bootstrap BCa por conglomerados).
- **Potencia.** La guía sugiere repetir la validación externa. Con datos temporales no se pueden barajar los folds, y 28 días es todo lo que hay; la familia por negocio compensa en parte la baja potencia de las pruebas diarias.

## 20.2 Verificación de supuestos

In [2]:
filas = []
for metrica, L in [("RMSE log", lambda e: e ** 2), ("MAE log", lambda e: np.abs(e))]:
    d_fila = L(y - preds["ACS (original)"]) - L(y - preds[RIVAL])
    d = pd.Series(d_fila).groupby(sub["fecha"].to_numpy()).mean()
    adf = adfuller(d, autolag="AIC")[1]
    sw = stats.shapiro(d).pvalue
    ac1 = d.autocorr(1)
    neg = pd.Series(d_fila).groupby(sub["negocio"].to_numpy()).mean()
    filas.append({"métrica": metrica, "ADF p (diferencia diaria estacionaria si < 0,05)": adf,
                  "Shapiro p (normalidad diaria)": sw, "autocorrelación rezago 1": ac1,
                  "asimetría de la diferencia por negocio": stats.skew(neg),
                  "días": len(d), "negocios": len(neg)})
supuestos = pd.DataFrame(filas).set_index("métrica")
supuestos.round(4)

,"ADF p (diferencia diaria estacionaria si < 0,05)",Shapiro p (normalidad diaria),autocorrelación rezago 1,asimetría de la diferencia por negocio,días,negocios
métrica,,,,,,
RMSE log,0.01,0.86,-0.06,7.46,28,447
MAE log,0.00,0.38,0.14,-2.68,28,447


**Lectura de los supuestos.**

- **Estacionariedad de la diferencia de pérdida** (supuesto de Diebold-Mariano, Giacomini-White, Clark-West, MCS y SPA): el ADF la rechaza en las dos métricas (p = 0,01 y < 0,01), así que la diferencia diaria es estacionaria.
- **Normalidad de la diferencia diaria:** Shapiro no rechaza (p = 0,86 y 0,38). Respalda la t de Student de la corrección HLN y la d de Cohen.
- **Autocorrelación de la diferencia:** es baja (−0,06 y 0,14 en el rezago 1). El DM con h = 1 es adecuado.
- **Simetría para Wilcoxon:** la diferencia por negocio es muy asimétrica (7,5 en RMSE), por los pocos negocios con días sin venta. Por eso se acompaña con la permutación, que no supone simetría, y con la delta de Cliff.
- **Anidamiento:** el ACS contiene a Random Forest como caso particular, así que se usa Clark-West además de Giacomini-White.

## 20.3 Tabla resumen

Media ± desviación de los 4 folds externos, intervalo de confianza del 95 % del RMSE log (bootstrap BCa por negocios, que respeta la dependencia dentro de cada negocio) y tiempos de entrenamiento e inferencia del modelo final.

In [3]:
from estadistica import ic_bca_negocios
res = pd.read_csv("resultados/original_resumen.csv", index_col=0)
lid = pd.read_csv("../data/lideres.csv")
nombres = ["ACS (original)"] + [f"{m} · {o}" for m, o in zip(lid["modelo"], lid["optimizador"]) if not m.startswith("ACS")] + ["Ingenuo estacional"]
neg = sub["negocio"].to_numpy()
filas = []
for n in nombres:
    e = y - preds[n]
    lo, hi = ic_bca_negocios(neg, lambda i, e=e: np.sqrt(np.mean(e[i] ** 2)), reps=500)
    r_ = res.loc[n]
    filas.append({"modelo": n, "RMSE log": f"{r_['RMSE log media']:.4f} ± {r_['RMSE log desv']:.4f}", "IC 95 % RMSE log (BCa)": f"[{lo:.3f}, {hi:.3f}]",
                  "MAE log": f"{r_['MAE log media']:.4f} ± {r_['MAE log desv']:.4f}",
                  "seg entrenamiento": r_.get("seg entrenamiento", np.nan), "seg inferencia (test)": r_.get("seg inferencia (test)", np.nan),
                  "_o": r_["RMSE log media"]})
resumen = pd.DataFrame(filas).sort_values("_o").drop(columns="_o").set_index("modelo")
resumen.round(3)

,RMSE log,IC 95 % RMSE log (BCa),MAE log,seg entrenamiento,seg inferencia (test)
modelo,,,,,
Random Forest · Grid,0.8874 ± 0.1266,"[0.752, 1.067]",0.2906 ± 0.0505,11.33,0.28
XGBoost · Random,0.8944 ± 0.1222,"[0.755, 1.076]",0.2961 ± 0.0479,0.60,0.04
ACS (original),0.8981 ± 0.1268,"[0.749, 1.085]",0.2878 ± 0.0511,71.88,0.35
HistGradientBoosting · Random,0.9047 ± 0.1346,"[0.770, 1.089]",0.3071 ± 0.0436,0.26,0.02
KNN · Genético,0.9314 ± 0.1112,"[0.780, 1.122]",0.3050 ± 0.0489,0.04,0.82
Lasso · Bayes,0.9493 ± 0.0894,"[0.799, 1.159]",0.3040 ± 0.0368,0.12,0.01
Ridge · Genético,0.9498 ± 0.0889,"[0.799, 1.159]",0.3038 ± 0.0352,0.03,0.00
Árbol · Bayes,0.9516 ± 0.1143,"[0.807, 1.142]",0.3324 ± 0.0414,0.23,0.01
SVR lineal · Grid,0.9883 ± 0.0963,"[0.826, 1.214]",0.3083 ± 0.0272,0.08,0.01


**Lectura.**

- **En RMSE log** el ACS queda **3.º** (0,8981 ± 0,1268), detrás de Random Forest (0,8874) y XGBoost (0,8944). Los intervalos BCa de los tres se solapan casi por completo.
- **En MAE log** es el **1.º** (0,2878 contra 0,2906 de Random Forest).
- **Costo:** entrenar el ACS toma 72 s, unas 6 veces Random Forest, porque entrena los 4 expertos 5 veces. La predicción de test toma 0,35 s, casi lo mismo que Random Forest.

## 20.4 Pruebas de hipótesis

Seis pruebas por métrica, del ACS contra el mejor benchmark (Random Forest · Grid), más Clark-West por el anidamiento: el ACS **contiene** a Random Forest como caso particular (pesos $w = e_{\text{RF}}$ en todas las hojas), que es exactamente el caso para el que Clark y West (2007) corrigen el sesgo del Diebold-Mariano. Los p-valores se ajustan con Holm y con Benjamini-Hochberg **dentro de cada métrica** (7 comparaciones por métrica, 14 en total).

In [4]:
def clark_west_diario(y, f_restr, f_amplio, dias):
    f = (y - f_restr) ** 2 - ((y - f_amplio) ** 2 - (f_restr - f_amplio) ** 2)
    fd = pd.Series(f).groupby(dias).mean()
    t = fd.mean() / (fd.std(ddof=1) / np.sqrt(len(fd)))
    return t, stats.t.sf(t, len(fd) - 1)


tablas = []
for metrica in ["rmse", "mae"]:
    t, mcs, dia = pruebas(y, sub, preds, RIVAL, metrica)
    if metrica == "rmse":
        cw, p = clark_west_diario(y, preds[RIVAL], preds["ACS (original)"], sub["fecha"].to_numpy())
        t = pd.concat([t, pd.DataFrame([{"prueba": f"Clark-West (RF anidado en el ACS)", "H0": "el ACS no mejora al modelo anidado (RF)",
                                         "supuestos": "modelos anidados; pérdida cuadrática diaria", "estadístico": cw, "p crudo": p,
                                         "métrica": "RMSE log"}])], ignore_index=True)
        t["p Holm"] = multipletests(t["p crudo"].fillna(1), method="holm")[1]
        t["p BH"] = multipletests(t["p crudo"].fillna(1), method="fdr_bh")[1]
    tablas.append(t)
pruebas_tab = pd.concat(tablas, ignore_index=True)
pruebas_tab["prueba"] = pruebas_tab["prueba"].str.replace(f" vs {RIVAL}", "", regex=False)
pruebas_tab["rechaza H0 (ajustado)"] = np.where(pruebas_tab["prueba"].str.startswith("Model Confidence"), "—",
                                                np.where(pruebas_tab["p Holm"] < 0.05, "sí", "no"))
cols = ["métrica", "prueba", "H0", "supuestos", "estadístico", "p crudo", "p Holm", "p BH", "d de Cohen", "delta de Cliff",
        "magnitud", "ACS mejor en (% negocios)", "IC 95 % (bootstrap estacionario)", "rechaza H0 (ajustado)"]
pd.set_option("display.max_colwidth", 60)
pruebas_tab[cols].round(4)

,métrica,prueba,H0,supuestos,estadístico,p crudo,p Holm,p BH,d de Cohen,delta de Cliff,magnitud,ACS mejor en (% negocios),IC 95 % (bootstrap estacionario),rechaza H0 (ajustado)
0,RMSE log,Model Confidence Set (35 modelos),el ACS pertenece al conjunto de los mejores (igual pérdi...,pérdida diaria estacionaria; bootstrap por bloques de 3 ...,NaN,0.57,1.00,0.79,NaN,NaN,NaN,NaN,NaN,—
1,RMSE log,Diebold-Mariano (HLN),igual pérdida esperada (bilateral),"diferencia de pérdida estacionaria (ADF), h = 1",1.29,0.21,1.00,0.49,0.24,NaN,NaN,NaN,"[-0.0064, 0.0446]",no
2,RMSE log,Giacomini-White,igual habilidad predictiva condicional,instrumentos: constante y diferencia del día anterior,2.23,0.33,1.00,0.57,0.24,NaN,NaN,NaN,NaN,no
3,RMSE log,SPA de Hansen (benchmark Random Forest · Grid),el ACS no es mejor que el benchmark,"bootstrap estacionario, bloques de 3",NaN,0.92,1.00,0.92,NaN,NaN,NaN,NaN,NaN,no
4,RMSE log,Wilcoxon por negocio,mediana de la diferencia por negocio = 0,negocios independientes; diferencias simétricas,"45,564.00",0.10,0.60,0.35,NaN,0.00,despreciable,44.97,NaN,no
5,RMSE log,Permutación pareada por negocio,intercambiabilidad de las pérdidas,negocios independientes,0.00,0.73,1.00,0.85,NaN,NaN,NaN,NaN,NaN,no
6,RMSE log,Clark-West (RF anidado en el ACS),el ACS no mejora al modelo anidado (RF),modelos anidados; pérdida cuadrática diaria,1.73,0.05,0.33,0.33,NaN,NaN,NaN,NaN,NaN,no
7,MAE log,Model Confidence Set (35 modelos),el ACS pertenece al conjunto de los mejores (igual pérdi...,pérdida diaria estacionaria; bootstrap por bloques de 3 ...,NaN,1.00,1.00,1.00,NaN,NaN,NaN,NaN,NaN,—
8,MAE log,Diebold-Mariano (HLN),igual pérdida esperada (bilateral),"diferencia de pérdida estacionaria (ADF), h = 1",-1.05,0.30,1.00,0.48,-0.20,NaN,NaN,NaN,"[-0.0077, 0.0027]",no
9,MAE log,Giacomini-White,igual habilidad predictiva condicional,instrumentos: constante y diferencia del día anterior,2.23,0.33,1.00,0.48,-0.20,NaN,NaN,NaN,NaN,no


**Lectura de las pruebas.**

- **RMSE log (métrica principal): ninguna prueba rechaza.**
  - El MCS mantiene al ACS (p = 0,57) junto con otros 10 modelos.
  - Diebold-Mariano (p = 0,21), Giacomini-White (0,33) y SPA (0,92) no rechazan.
  - Clark-West, la prueba indicada por el anidamiento, da t = 1,73, que no sobrevive al ajuste de Holm.
  - Por negocio, el ACS es mejor en solo el 45 % de los negocios (Wilcoxon p = 0,10; delta de Cliff 0,005, despreciable).
- **MAE log (secundaria): tampoco rechaza ninguna.** El ACS tiene la menor media, pero la diferencia (d = −0,20) no es significativa con 28 días, y el MCS conserva 19 modelos.
- En total se hicieron **13 comparaciones**: 7 en RMSE y 6 en MAE (11 pareadas y 2 MCS). Ninguna pareada rechaza al 5 % ni siquiera sin ajustar, salvo Clark-West, así que el tipo de ajuste no cambia la conclusión.

### Model Confidence Set con el ACS incluido

In [5]:
mcs_r = pd.read_csv("resultados/original_mcs_rmse.csv", index_col=0)
mcs_m = pd.read_csv("resultados/original_mcs_mae.csv", index_col=0)
mcs = pd.concat({"RMSE log": mcs_r[["p MCS", "en el conjunto"]], "MAE log": mcs_m[["p MCS", "en el conjunto"]]}, axis=1)
mcs = mcs.loc[mcs_r.index]
print(f"en el conjunto al 90 %: RMSE log {int(mcs_r['en el conjunto'].sum())} modelos; MAE log {int(mcs_m['en el conjunto'].sum())} modelo(s)")
mcs.head(14).round(3)

en el conjunto al 90 %: RMSE log 11 modelos; MAE log 19 modelo(s)


RMSE log                MAE log               
                                   p MCS en el conjunto   p MCS en el conjunto
Model name                                                                    
Random Forest · Grid                1.00           True    0.28           True
Random Forest · Random              0.57           True    0.24           True
Random Forest · Bayes               0.57           True    0.24           True
XGBoost · Random                    0.53           True    0.07          False
ACS (original)                      0.57           True    1.00           True
XGBoost · Genético                  0.49           True    0.04          False
Random Forest · Genético            0.57           True    0.24           True
XGBoost · Bayes                     0.57           True    0.24           True
HistGradientBoosting · Random       0.12           True    0.00          False
HistGradientBoosting · Bayes        0.22           True    0.00          False
HistGradientBoosting · Grid         0.01          False    0.00          False
XGBoost · Grid                      0.12           True    0.18           True
HistGradientBoosting · Genético     0.02          False    0.00          False
KNN · Genético                      0.01          False    0.00          False

### Confirmación en test (3 – 23 sep)

El test no se usó para elegir nada; se usa aquí solo como confirmación independiente (21 días).

In [6]:
pt = pd.read_csv("../data/pred_test.csv.gz", parse_dates=["fecha"])
yt = a_log(pt["venta_total"].to_numpy())
filas = []
for metrica, L in [("RMSE log", lambda e: e ** 2), ("MAE log", lambda e: np.abs(e))]:
    d = pd.Series(L(yt - a_log(pt["acs"])) - L(yt - a_log(pt["rf"]))).groupby(pt["fecha"].to_numpy()).mean().to_numpy()
    dm, p = diebold_mariano(d)
    lo, hi = ic_estacionario(d)
    m_acs = np.sqrt(np.mean((yt - a_log(pt["acs"])) ** 2)) if metrica == "RMSE log" else np.mean(np.abs(yt - a_log(pt["acs"])))
    m_rf = np.sqrt(np.mean((yt - a_log(pt["rf"])) ** 2)) if metrica == "RMSE log" else np.mean(np.abs(yt - a_log(pt["rf"])))
    filas.append({"métrica": metrica, "ACS": m_acs, "Random Forest": m_rf, "DM (HLN)": dm, "p": p,
                  "IC 95 % diferencia diaria": f"[{lo:.4f}, {hi:.4f}]", "d de Cohen": cohen_d(d)})
pd.DataFrame(filas).set_index("métrica").round(4)

,ACS,Random Forest,DM (HLN),p,IC 95 % diferencia diaria,d de Cohen
métrica,,,,,,
RMSE log,0.61,0.62,-1.85,0.08,"[-0.0287, -0.0006]",-0.40
MAE log,0.21,0.22,-3.50,0.00,"[-0.0078, -0.0028]",-0.76


**Lectura de la confirmación en test.** Aquí el ACS sí mejora a Random Forest:

- **RMSE log:** 0,611 contra 0,622. El intervalo del bootstrap estacionario de la diferencia diaria **excluye el 0** ([−0,029; −0,001]), aunque el DM queda en p = 0,08 (d = −0,40).
- **MAE log:** DM p < 0,01, con d = −0,76.

```{warning}
**Este resultado no cuenta para R3**, por dos razones:

1. El test (21 días) **no es la evaluación que fija la guía**: esa es el bucle externo.
2. **El test también se miró durante la iteración del diseño.** El script de cada versión imprimía su RMSE de test (0,618, 0,621, 0,621 y 0,611), y la v4 se eligió después de verlo. Por eso es una comprobación contaminada.

Que el test favorezca al ACS mientras el bucle externo favorece a Random Forest dice, sobre todo, que **la diferencia entre los dos está en el nivel del ruido** de un trimestre de datos.
```

## 20.5 Evaluación en tres conjuntos de datos adicionales

Mismo diseño (85 días por entidad: 14 de historia, 50 de train, 21 de test), mismos folds externos (4 × 7 días), misma semilla y **mismo presupuesto** (16 evaluaciones bayesianas) para los 8 benchmarks; el ACS usa como expertos los mismos 4 modelos, con los hiperparámetros elegidos en cada fold de **ese** conjunto.

| Conjunto | Naturaleza | Entidades | Particularidad |
|---|---|---|---|
| M5 (Walmart; Makridakis et al., 2022) | ventas minoristas por tienda × departamento | 70 | sin ceros, muy regular |
| Online Retail II (UCI; Chen, 2019) | ingreso diario de comercio electrónico por producto | 60 | 19 % de días sin venta, devoluciones negativas |
| ElectricityLoadDiagrams (UCI; Trindade, 2015) | consumo diario de energía por cliente | 368 | series suaves, escalas muy distintas |

In [7]:
from experimentos_externos import CONJUNTOS, MODELOS, EXT, cargar_conjunto
NOMBRE = {"knn": "KNN", "ridge": "Ridge", "lasso": "Lasso", "arbol": "Árbol", "rf": "Random Forest", "xgb": "XGBoost",
          "svr": "SVR lineal", "hgb": "HistGradientBoosting"}
filas, rangos = [], {}
for c in CONJUNTOS:
    tr, _ = cargar_conjunto(c)
    yl = a_log(tr["venta_total"].to_numpy())
    met = {}
    for m in MODELOS:
        rr = pickle.load(open(EXT / "resultados" / f"{c}__{m}__bayes.pkl", "rb"))
        met[NOMBRE[m]] = (np.mean([f["metricas"]["rmse_log"] for f in rr["folds"]]), np.mean([f["metricas"]["mae_log"] for f in rr["folds"]]))
    ac = pickle.load(open(EXT / "resultados" / f"{c}__acs.pkl", "rb"))
    rm = [np.sqrt(np.mean((yl[f["pred"]["fila"]] - f["pred"]["ACS"]) ** 2)) for f in ac]
    ma = [np.mean(np.abs(yl[f["pred"]["fila"]] - f["pred"]["ACS"])) for f in ac]
    met["ACS (original)"] = (np.mean(rm), np.mean(ma))
    t = pd.DataFrame(met, index=["RMSE log", "MAE log"]).T
    rangos[c] = t["RMSE log"].rank()
    for n, v in t.iterrows():
        filas.append({"conjunto": c, "modelo": n, "RMSE log": v["RMSE log"], "MAE log": v["MAE log"],
                      "rango RMSE": int(t["RMSE log"].rank()[n]), "rango MAE": int(t["MAE log"].rank()[n])})
ext = pd.DataFrame(filas)
ext.pivot(index="modelo", columns="conjunto", values="rango RMSE").assign(
    **{"rango medio": lambda d: d.mean(axis=1)}).sort_values("rango medio")

conjunto,electricidad,m5,retail,rango medio
modelo,,,,
ACS (original),2,2,4,2.67
SVR lineal,6,1,1,2.67
Ridge,4,3,3,3.33
Random Forest,1,5,6,4.00
Lasso,7,4,2,4.33
XGBoost,5,6,5,5.33
HistGradientBoosting,3,7,7,5.67
KNN,8,8,8,8.00
Árbol,9,9,9,9.00


In [8]:
ext[ext["modelo"].isin(["ACS (original)", "Random Forest", "Ridge", "SVR lineal"])].pivot(
    index="modelo", columns="conjunto", values=["RMSE log", "MAE log"]).round(4)

RMSE log                  MAE log            
conjunto       electricidad   m5 retail electricidad   m5 retail
modelo                                                          
ACS (original)         0.22 0.18   1.27         0.06 0.13   0.89
Random Forest          0.22 0.18   1.30         0.06 0.13   0.89
Ridge                  0.22 0.18   1.27         0.07 0.13   0.90
SVR lineal             0.23 0.18   1.25         0.08 0.13   0.87

In [9]:
# Friedman sobre los 4 conjuntos (Masser + 3 externos), con el rango de RMSE log de cada modelo en cada conjunto
r_m = res.loc[["ACS (original)"] + [f"{NOMBRE[m]} · {o}" for m, o in zip(
    ["knn", "ridge", "lasso", "arbol", "rf", "xgb", "svr", "hgb"], ["Genético", "Genético", "Bayes", "Bayes", "Grid", "Random", "Grid", "Random"])],
    "RMSE log media"]
r_m.index = ["ACS (original)"] + [NOMBRE[m] for m in ["knn", "ridge", "lasso", "arbol", "rf", "xgb", "svr", "hgb"]]
tabla_f = pd.DataFrame({"masser": r_m, **{c: ext[ext["conjunto"] == c].set_index("modelo")["RMSE log"] for c in CONJUNTOS}}).T
fn = friedman_nemenyi(tabla_f)
print(f"Friedman con 4 conjuntos y 9 modelos: χ² = {fn['chi2']:.2f}, p = {fn['p']:.3f}; CD de Nemenyi = {fn['cd']:.2f}")
fn["rangos"].round(2).to_frame("rango medio")

Friedman con 4 conjuntos y 9 modelos: χ² = 15.33, p = 0.053; CD de Nemenyi = 6.01


,rango medio
ACS (original),2.75
Random Forest,3.25
Ridge,4.25
SVR lineal,4.25
XGBoost,4.50
Lasso,4.75
HistGradientBoosting,5.25
KNN,7.25
Árbol,8.75


**Lectura de los conjuntos externos** (no se usaron para diseñar ninguna versión del ACS):

- **El ACS no es el mejor en ningún conjunto externo:** es 2.º en Electricity y en M5 y 4.º en Online Retail. El mejor es Random Forest en Electricity y el SVR lineal en M5 y Retail.
- **Pero es el más consistente.** Sumando Masser, tiene el **mejor rango medio** (2,75, contra 3,25 de Random Forest). El modelo que gana en un conjunto pierde en otro:
  - Random Forest es 1.º en Electricity y Masser, pero 5.º y 6.º en M5 y Retail.
  - El SVR lineal es 1.º en M5 y Retail, pero 6.º en Electricity.

  Es el comportamiento esperable de una combinación: **robusta, pero no la mejor en cada caso**.
- **Friedman con 4 conjuntos no rechaza** (p = 0,053) y la diferencia crítica de Nemenyi (6,0 rangos) es enorme. Con 4 conjuntos no se puede afirmar que el ACS supere a ningún modelo en general; Demšar (2006) recomienda al menos 10.
- Los mejores modelos en M5 y Retail son lineales y el SVR **no está** entre los expertos del ACS. Los expertos se fijaron de antemano con los resultados del panel de Masser; agregarlo ahora sería ajustar el modelo a estos datos.

## 20.6 Veredicto sobre R3

In [10]:
p_rmse = pruebas_tab[pruebas_tab["métrica"] == "RMSE log"]
p_mae = pruebas_tab[pruebas_tab["métrica"] == "MAE log"]
bench = res.drop(index=["ACS (original)", "Ingenuo estacional", "Persistencia"])
criterios = pd.DataFrame([
    {"criterio (guía, sec. 6.1)": "1. Media del bucle externo mejor que el mejor benchmark y que los 7 modelos",
     "RMSE log (principal)": "no" if res.loc["ACS (original)", "RMSE log media"] >= bench["RMSE log media"].min() else "sí",
     "MAE log (secundaria)": "no" if res.loc["ACS (original)", "MAE log media"] >= bench["MAE log media"].min() else "sí"},
    {"criterio (guía, sec. 6.1)": "2. Al menos 3 pruebas distintas rechazan con p ajustado < 0,05 (MCS: ACS único en el conjunto)",
     "RMSE log (principal)": f"{int((p_rmse['rechaza H0 (ajustado)'] == 'sí').sum())} de {int((p_rmse['rechaza H0 (ajustado)'] != '—').sum())} pruebas pareadas; MCS con {int(mcs_r['en el conjunto'].sum())} modelos",
     "MAE log (secundaria)": f"{int((p_mae['rechaza H0 (ajustado)'] == 'sí').sum())} de {int((p_mae['rechaza H0 (ajustado)'] != '—').sum())} pruebas pareadas; MCS con {int(mcs_m['en el conjunto'].sum())} modelo"},
    {"criterio (guía, sec. 6.1)": "3. Efecto no despreciable (|δ| ≥ 0,147 o |d| ≥ 0,2)",
     "RMSE log (principal)": f"d = {p_rmse['d de Cohen'].dropna().iloc[0]:.2f}; δ = {p_rmse['delta de Cliff'].dropna().iloc[0]:.3f}",
     "MAE log (secundaria)": f"d = {p_mae['d de Cohen'].dropna().iloc[0]:.2f}; δ = {p_mae['delta de Cliff'].dropna().iloc[0]:.3f}"},
    {"criterio (guía, sec. 6.1)": "Generalización: mejor en los 3 conjuntos externos",
     "RMSE log (principal)": "no", "MAE log (secundaria)": "no"},
]).set_index("criterio (guía, sec. 6.1)")
criterios

,RMSE log (principal),MAE log (secundaria)
"criterio (guía, sec. 6.1)",,
1. Media del bucle externo mejor que el mejor benchmark y que los 7 modelos,no,sí
"2. Al menos 3 pruebas distintas rechazan con p ajustado < 0,05 (MCS: ACS único en el conjunto)",0 de 6 pruebas pareadas; MCS con 11 modelos,0 de 5 pruebas pareadas; MCS con 19 modelo
"3. Efecto no despreciable (|δ| ≥ 0,147 o |d| ≥ 0,2)",d = 0.24; δ = 0.005,d = -0.20; δ = 0.008
Generalización: mejor en los 3 conjuntos externos,no,no


### Veredicto

```{important}
**R3 no se cumple.** En la métrica principal (RMSE log, fijada en la entrega 2 antes de diseñar el modelo), el ACS **no supera** a Random Forest:

- su media del bucle externo es 0,011 peor;
- **ninguna** de las 6 pruebas pareadas rechaza, y tampoco ninguna de las 5 en MAE log;
- el efecto es pequeño o despreciable;
- en los conjuntos externos no es el mejor en ninguno.

El tope de la nota según la guía es **4,0**.
```

**Lo que sí muestra la evidencia, sin que alcance para R3:**

1. El ACS está siempre en el **grupo de los mejores**: en el MCS de las dos métricas, con el menor MAE medio y con intervalos que se solapan con los de Random Forest.
2. Es el **más consistente entre conjuntos**: tiene el mejor rango medio en los 4.
3. **Mejora a Random Forest en test**, pero esa comparación está contaminada por la iteración del diseño.

**Causas probables de la no superioridad:**

1. **Información.** El error dominante (días sin venta súbitos) no tiene señal en el día anterior, y ningún combinador de expertos que fallan igual lo puede corregir.
2. **Expertos muy correlacionados.** La ganancia posible de combinar es pequeña: el puzzle de la combinación (Smith y Wallis, 2009).
3. **El combinador sobreajusta** con pocos datos (19.9 y 19.11).
4. **Potencia.** Con 28 días de validación externa, diferencias de 1 % en RMSE no son detectables.